# EDA: PJM DA LMP spikes and weather

Placeholder notebook. Run the pipeline first (`./scripts/run_smoke.sh` or `./scripts/run_mvp.sh`) so that `data/processed/panel_comed.parquet` exists.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ep_spikes import paths
panel = pd.read_parquet(paths.PROCESSED_DIR / 'panel_comed.parquet')
print('Panel shape:', panel.shape)
panel.head()

In [ ]:
# Price distribution
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(panel['lmp'].dropna(), bins=80)
ax[0].set_title('DA LMP histogram')
ax[0].set_xlabel('$/MWh')
panel.groupby('season_name')['lmp'].plot(kind='kde', ax=ax[1], legend=True)
ax[1].set_xlim(-50, 500)
ax[1].set_title('KDE by season')
plt.tight_layout(); plt.show()

In [ ]:
# Spike counts per month
panel['is_spike_abs'] = (panel['lmp'] > 300).astype(int)
monthly = panel.groupby([panel.index.year, panel.index.month])['is_spike_abs'].sum()
monthly.plot(kind='bar', figsize=(12, 3))
plt.title('Count of hours with LMP > $300 per month')
plt.ylabel('Spike hours')
plt.tight_layout(); plt.show()

In [ ]:
# Weather-price scatter (summer only)
summer = panel[panel['season_name'] == 'summer']
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(summer['temperature_2m'], summer['lmp'], s=3, alpha=0.3)
ax.set_xlabel('temperature 2m (°C)'); ax.set_ylabel('DA LMP ($/MWh)')
ax.set_title('Summer: DA LMP vs temperature')
ax.set_ylim(0, 500)
plt.tight_layout(); plt.show()